#### Train a model to distinguish colors

In [13]:
# Load necessary libraries
import csv
import os
import sys
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from ultralytics import YOLO

In [14]:
# Load the indexed color dataset
dataset_csv_path = Path("color_dataset_index.csv").resolve()
color_df = pd.read_csv(dataset_csv_path)

# Keep only valid rows with existing image files
color_df = color_df.dropna(subset=["image_path", "color"]).copy()
color_df["image_path"] = color_df["image_path"].astype(str)
color_df["color"] = color_df["color"].astype(str).str.lower()
color_df = color_df[color_df["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

if color_df.empty:
    raise ValueError("No valid image rows found in color_dataset_index.csv")

# Stratified split so each color appears in train and val
train_df, val_df = train_test_split(
    color_df,
    test_size=0.2,
    random_state=42,
    stratify=color_df["color"]
 )

print(f"Total valid images: {len(color_df)}")
print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")
print("Classes:", sorted(color_df["color"].unique()))

Total valid images: 448
Train images: 358
Val images: 90
Classes: ['black', 'blue', 'brown', 'green', 'orange', 'red', 'violet', 'white', 'yellow']


In [15]:
import shutil

# Build YOLO classification folder structure:
# color_yolo_dataset/
#   train/<class_name>/*.jpg
#   val/<class_name>/*.jpg
yolo_color_root = Path("color_yolo_dataset").resolve()
if yolo_color_root.exists():
    shutil.rmtree(yolo_color_root)

for split_name in ["train", "val"]:
    split_df = train_df if split_name == "train" else val_df
    for _, row in split_df.iterrows():
        class_dir = yolo_color_root / split_name / row["color"]
        class_dir.mkdir(parents=True, exist_ok=True)

        src_path = Path(row["image_path"])
        # Add index prefix to avoid filename collisions across source folders
        dst_name = f"{len(list(class_dir.glob('*'))):06d}_{src_path.name}"
        shutil.copy2(src_path, class_dir / dst_name)

print(f"YOLO dataset created at: {yolo_color_root}")
print("Train class folders:", len(list((yolo_color_root / 'train').glob('*'))))
print("Val class folders:", len(list((yolo_color_root / 'val').glob('*'))))

YOLO dataset created at: D:\VMGP\monash\2026 unihack\TayUnihack2026\color_yolo_dataset
Train class folders: 9
Val class folders: 9


In [16]:
# Train YOLOv8 classifier on GPU if CUDA is available
device = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("CUDA enabled:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available, training on CPU")

# You can switch to yolov8l-cls.pt if you want higher capacity
model = YOLO("yolov8m-cls.pt")

color_train = model.train(
    data=str(yolo_color_root),
    epochs=30,
    imgsz=224,
    batch=32,
    device=device,
    workers=4,
    project="runs_color_classifier",
    name="yolov8m_color",
    exist_ok=True
)

print("Training finished.")
color_train

CUDA enabled: NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics 8.4.21  Python-3.14.3 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\VMGP\monash\2026 unihack\TayUnihack2026\color_yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_co

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000016449001D30>
curves: []
curves_results: []
fitness: 0.9277777671813965
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8888888955116272, 'metrics/accuracy_top5': 0.9666666388511658, 'fitness': 0.9277777671813965}
save_dir: WindowsPath('D:/VMGP/monash/2026 unihack/TayUnihack2026/runs/classify/runs_color_classifier/yolov8m_color')
speed: {'preprocess': 0.07864999998774794, 'inference': 0.5840899999586529, 'loss': 0.00014222217335676152, 'postprocess': 0.00026666667609889474}
task: 'classify'
top1: 0.8888888955116272
top5: 0.9666666388511658

#### Train a model to distinguish different clothes types

In [17]:
# Load the indexed clothes dataset
dataset_csv_path = Path("clothes_dataset_index.csv").resolve()
clothes_df = pd.read_csv(dataset_csv_path)

# Keep only valid rows with existing image files
clothes_df = clothes_df.dropna(subset=["image_path", "category"]).copy()
clothes_df["image_path"] = clothes_df["image_path"].astype(str)
clothes_df["category"] = clothes_df["category"].astype(str).str.lower()
clothes_df = clothes_df[clothes_df["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

if clothes_df.empty:
    raise ValueError("No valid image rows found in clothes_dataset_index.csv")

# Stratified split so each category appears in train and val
train_df, val_df = train_test_split(
    clothes_df,
    test_size=0.2,
    random_state=42,
    stratify=clothes_df["category"]
 )

print(f"Total valid images: {len(clothes_df)}")
print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")
print("Classes:", sorted(clothes_df["category"].unique()))

Total valid images: 7391
Train images: 5912
Val images: 1479
Classes: ['coat', 'dress', 'hoodie', 'jeans', 'pants', 'shirt', 'shorts', 'skirt', 'sweater', 't-shirt']


In [18]:
# Build YOLO classification folder structure:
# category_yolo_dataset/
#   train/<class_name>/*.jpg
#   val/<class_name>/*.jpg
yolo_category_root = Path("category_yolo_dataset").resolve()
if yolo_category_root.exists():
    shutil.rmtree(yolo_category_root)

for split_name in ["train", "val"]:
    split_df = train_df if split_name == "train" else val_df
    for _, row in split_df.iterrows():
        class_dir = yolo_category_root / split_name / row["category"]
        class_dir.mkdir(parents=True, exist_ok=True)

        src_path = Path(row["image_path"])
        # Add index prefix to avoid filename collisions across source folders
        dst_name = f"{len(list(class_dir.glob('*'))):06d}_{src_path.name}"
        shutil.copy2(src_path, class_dir / dst_name)

print(f"YOLO dataset created at: {yolo_category_root}")
print("Train class folders:", len(list((yolo_category_root / 'train').glob('*'))))
print("Val class folders:", len(list((yolo_category_root / 'val').glob('*'))))

YOLO dataset created at: D:\VMGP\monash\2026 unihack\TayUnihack2026\category_yolo_dataset
Train class folders: 10
Val class folders: 10


In [19]:
# Train YOLOv8 classifier on GPU if CUDA is available
device = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("CUDA enabled:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available, training on CPU")

# You can switch to yolov8l-cls.pt if you want higher capacity
model = YOLO("yolov8n-cls.pt")

category_train = model.train(
    data=str(yolo_category_root),
    epochs=30,
    imgsz=224,
    batch=32,
    device=device,
    workers=4,
    project="runs_category_classifier",
    name="yolov8n_category",
    exist_ok=True
)

print("Training finished.")
category_train

CUDA enabled: NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics 8.4.21  Python-3.14.3 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\VMGP\monash\2026 unihack\TayUnihack2026\category_yolo_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000164866B0E50>
curves: []
curves_results: []
fitness: 0.9347532093524933
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8782961368560791, 'metrics/accuracy_top5': 0.9912102818489075, 'fitness': 0.9347532093524933}
save_dir: WindowsPath('D:/VMGP/monash/2026 unihack/TayUnihack2026/runs/classify/runs_category_classifier/yolov8n_category')
speed: {'preprocess': 0.07969810683591537, 'inference': 0.14575611900175658, 'loss': 0.00010256930977371197, 'postprocess': 0.0002163623991062483}
task: 'classify'
top1: 0.8782961368560791
top5: 0.9912102818489075